# Part 2 : Transfer learning ( Using an existing model )

we are training the model to Classify aircraft images into aircraft **variants**. (e.g., Boeing 737-800, Airbus A320-214)

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

## Load data :

In [ ]:
# transformation pipeline :
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Download data
train_dataset = torchvision.datasets.FGVCAircraft(root="data/fgvc_aircraft", split="train", transform=train_transforms, download=True)
val_dataset   = torchvision.datasets.FGVCAircraft(root="data/fgvc_aircraft", split="val", transform=transform, download=True)
test_dataset  = torchvision.datasets.FGVCAircraft(root="data/fgvc_aircraft", split="test", transform=transform, download=True)


# Load dataset :
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

100%|██████████| 2.75G/2.75G [03:00<00:00, 15.2MB/s]


In [ ]:
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Validation size: {len(val_dataset)}")

Train size: 3334
Test size: 3333
Validation size: 3333


## Load the model **ResNet-50** :

In [ ]:
# Load ResNet-50 pretrained on ImageNet :
resnet50 = models.resnet34(pretrained=True)

In [ ]:
# Check the model architecture :
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Replace the final FC layer:

In [ ]:
# Replace in the FC layer
num_variants = 100
resnet50.fc = nn.Linear(in_features=512, out_features=num_variants)

## Train the model : transfer learning

In [ ]:
# Move the model to the GPU memory :
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet34 = resnet50.to(device)

In [ ]:
# Define the loss function and the optimizer :
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet34.parameters(), lr=1e-4)

Accuracy : the percentage of correctly classified images

In [ ]:
num_epochs = 10
final_acc = 0.0  # to store final accuracy

for epoch in range(num_epochs):
    resnet34.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = resnet34(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Calculate training accuracy for this batch
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    final_acc = epoch_acc  # update final accuracy

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

print(f"\nFinal Training Accuracy after {num_epochs} epochs: {final_acc:.2f}%")

Epoch [1/10], Loss: 4.1934, Accuracy: 9.48%
Epoch [2/10], Loss: 3.2203, Accuracy: 25.55%
Epoch [3/10], Loss: 2.6442, Accuracy: 37.55%
Epoch [4/10], Loss: 2.1835, Accuracy: 46.55%
Epoch [5/10], Loss: 1.8761, Accuracy: 53.36%
Epoch [6/10], Loss: 1.5864, Accuracy: 61.04%
Epoch [7/10], Loss: 1.3816, Accuracy: 65.87%
Epoch [8/10], Loss: 1.2487, Accuracy: 68.09%
Epoch [9/10], Loss: 1.1400, Accuracy: 70.55%
Epoch [10/10], Loss: 1.0291, Accuracy: 73.49%

Final Training Accuracy after 10 epochs: 73.49%


## Evaluate test data :

In [ ]:
def evaluate_model(model, dataloader, criterion, device, name="Dataset"):
    model.eval()  # evaluation mode

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # disable gradients
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = running_loss / len(dataloader)
    accuracy = 100 * correct / total

    return avg_loss, accuracy

In [ ]:
test_loss, test_accuracy = evaluate_model(
    resnet34,
    test_loader,
    criterion,
    device,
    name="Test data"
)

print(f"Test loss : {test_loss}")
print(f"Test accuracy : {test_accuracy}")

Test loss : 1.802971302043824
Test accuracy : 50.28502850285029


In [ ]:
val_loss, val_accuracy = evaluate_model(
    resnet34,
    val_loader,
    criterion,
    device,
    name="Validation data"
)

print(f"Validation loss : {val_loss}")
print(f"Validation accuracy : {val_accuracy}")

Validation loss : 1.8325240502754847
Validation accuracy : 49.86498649864986


In [ ]:
import torch
from google.colab import files

# 1. Save your model locally in the Colab VM
torch.save(resnet34.state_dict(), 'my_model.pth')

# 2. Download it to your computer
files.download('my_model.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>